In [2]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier
)
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. قراءة البيانات
train_df = pd.read_parquet("../artifacts/train_processed.parquet")
val_df = pd.read_parquet("../artifacts/val_processed.parquet")
test_df = pd.read_parquet("../artifacts/test_processed.parquet")

X_train, y_train = train_df.drop(columns=["is_late"]), train_df["is_late"]
X_val, y_val = val_df.drop(columns=["is_late"]), val_df["is_late"]
X_test, y_test = test_df.drop(columns=["is_late"]), test_df["is_late"]

# 2. تعريف النماذج الـ 8 (بما فيها KNN والنماذج الشبيهة)
models = {
    "Dummy (Baseline)": DummyClassifier(strategy="most_frequent"),
    "KNN (n=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Naive Bayes": GaussianNB(),
    "Linear SVM": SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42),
    "Logistic Regression": LogisticRegression(class_weight="balanced", random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42, max_depth=10),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=42, n_estimators=100, max_depth=12, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42, n_estimators=100)
}

# 3. تدريب وتقييم كل نموذج
results = []
best_model = None
best_model_name = ""
best_val_auc = -1

print("--- بداية مقارنة النماذج الشاملة (8 نماذج) ---\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    
    # التنبؤ الاحتمالي
    if hasattr(model, "predict_proba"):
        val_probs = model.predict_proba(X_val)[:, 1]
    elif hasattr(model, "decision_function"):
        val_probs = model.decision_function(X_val)
    else:
        val_probs = model.predict(X_val)
        
    val_preds = model.predict(X_val)
    
    val_auc = roc_auc_score(y_val, val_probs)
    val_f1 = f1_score(y_val, val_preds, zero_division=0)
    
    results.append({
        "Model": name,
        "Val ROC-AUC": round(val_auc, 4),
        "Val F1-Score": round(val_f1, 4)
    })
    
    # اختيار الأفضل بناءً على Val ROC-AUC
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_model = model
        best_model_name = name

# عرض جدول المقارنة مرتباً حسب الأفضلية
results_df = pd.DataFrame(results).sort_values(by="Val ROC-AUC", ascending=False)
print(results_df.to_string(index=False))

print(f"\nالنموذج الأفضل على الـ Validation هو: **{best_model_name}** بـ ROC-AUC قدره: {best_val_auc:.4f}\n")

# 4. التقييم النهائي للنموذج الأفضل على الـ Test Set (مرة واحدة)
test_probs = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else best_model.predict(X_test)
test_preds = best_model.predict(X_test)

test_auc = roc_auc_score(y_test, test_probs)

print(f"--- التقييم النهائي للنموذج الأفضل ({best_model_name}) على الـ Test Set ---")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(classification_report(y_test, test_preds))

# 5. حفظ النموذج الأفضل والملخص
os.makedirs("../artifacts/models", exist_ok=True)
joblib.dump(best_model, "../artifacts/models/final_model.joblib")

summary = f"""
MLOps Task 2 - Extended Model Comparison Summary:
==================================================
{results_df.to_string(index=False)}

Best Model: {best_model_name}
Final Test ROC-AUC: {test_auc:.4f}
"""

with open("../artifacts/results_summary.txt", "w") as f:
    f.write(summary)

print("تم حفظ النموذج الأفضل والتقرير النهائي بنجاح!")

--- بداية مقارنة النماذج الشاملة (8 نماذج) ---

              Model  Val ROC-AUC  Val F1-Score
      Random Forest       0.7545        0.2963
  Gradient Boosting       0.7527        0.0034
      Decision Tree       0.7206        0.2443
Logistic Regression       0.6116        0.1815
         Linear SVM       0.6112        0.1800
        Naive Bayes       0.6027        0.0493
          KNN (n=5)       0.5981        0.0775
   Dummy (Baseline)       0.5000        0.0000

النموذج الأفضل على الـ Validation هو: **Random Forest** بـ ROC-AUC قدره: 0.7545

--- التقييم النهائي للنموذج الأفضل (Random Forest) على الـ Test Set ---
Test ROC-AUC: 0.7498
              precision    recall  f1-score   support

           0       0.95      0.80      0.87     13743
           1       0.19      0.55      0.28      1174

    accuracy                           0.78     14917
   macro avg       0.57      0.68      0.58     14917
weighted avg       0.89      0.78      0.82     14917

تم حفظ النموذج الأفضل والتق